# Transcribe Audio from Backblaze B2 with OpenAI Whisper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/backblaze-b2-samples/notebooks/blob/main/whisper-b2-transcription/whisper_b2_transcription.ipynb) [![Open In Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/backblaze-b2-samples/notebooks/HEAD?urlpath=lab/tree/whisper-b2-transcription/whisper_b2_transcription.ipynb) [![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/backblaze-b2-samples/notebooks?quickstart=1)

End-to-end notebook that streams an audio file from a public Backblaze B2 bucket, runs OpenAI Whisper to produce a transcript, and (optionally) writes the resulting JSON back to your own private B2 bucket.

## About OpenAI Whisper

[Whisper](https://github.com/openai/whisper) is OpenAI's open-source speech-recognition model. It supports 99 languages, is robust to background noise and accents, and ships several model sizes (`tiny`, `base`, `small`, `medium`, `large`) trading accuracy for latency. The `tiny` model used below runs in seconds on CPU and is fine for the ~10 second demo clip.

## About the Demo Audio

`jfk.flac` is the ~10 second clip from John F. Kennedy's inaugural address that ships with the openai/whisper repo as the canonical Whisper smoke-test asset. It is a US government recording in the public domain.

## Audio Hosted on Backblaze B2

| | |
|---|---|
| **Location** | `s3://b2datasets/whisper-demo/jfk.flac` |
| **Endpoint** | `https://s3.us-west-001.backblazeb2.com` |
| **Access** | Public (anonymous read); no application key needed for the input. |
| **Format** | FLAC, 16 kHz mono, ~10 s. |

## What This Notebook Does

1. Streams the audio from the public B2 bucket using `boto3`.
2. Loads OpenAI Whisper and transcribes the file.
3. (Optional) Writes the transcript JSON back to your own private B2 bucket using a second S3 client.

## Setup

Install Whisper + the S3 client:

In [ ]:
%pip install -q boto3 openai-whisper

## Configuration

### Reading from the Public Dataset Bucket

The demo audio lives in a Public B2 bucket so the read path needs no credentials. The endpoint and region for `b2datasets` are pinned explicitly below; do not rely on env vars for the public-read path because CI / other notebooks may have `AWS_ENDPOINT_URL_S3` set to a different region's endpoint (B2 returns an opaque 403 on signature/region mismatches).

In [ ]:
PUBLIC_ENDPOINT = "https://s3.us-west-001.backblazeb2.com"
PUBLIC_REGION = "us-west-001"
PUBLIC_BUCKET = "b2datasets"
PUBLIC_PREFIX = "whisper-demo"
AUDIO_KEY = f"{PUBLIC_PREFIX}/jfk.flac"

print(f"Will read s3://{PUBLIC_BUCKET}/{AUDIO_KEY} from {PUBLIC_ENDPOINT}")

### Your B2 Bucket for the Transcript Output (Optional)

To persist the transcript back to B2 you need an application key with **write** access to a bucket of your own. The notebook honors env vars first (Codespaces / CI / pre-exported shell), then falls back to an interactive prompt.

| Environment | Secret store | Notes |
|---|---|---|
| **GitHub Codespaces** &#11088; | User Settings &rarr; Codespaces &rarr; *Codespaces secrets* | Add `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` (or `B2_APPLICATION_KEY_ID` / `B2_APPLICATION_KEY`) scoped to this repo. Injected as env vars at container start. |
| **Google Colab** | Left sidebar &rarr; &#128273; *Secrets* | Read via `google.colab.userdata.get(...)`. |
| **Binder / local** | None / shell env | Falls back to `getpass.getpass()` below. |

Leave the bucket name blank (and don't set `PRIVATE_B2_BUCKET`) to skip the upload step entirely. The transcript is still printed.

In [ ]:
import getpass
import os


def _resolve_credential(env_var, *, b2_alias=None, prompt=""):
    """Populate env_var from existing env, Backblaze-named alias, Colab Secrets,
    then an interactive getpass prompt."""
    if os.environ.get(env_var):
        return
    if b2_alias and os.environ.get(b2_alias):
        os.environ[env_var] = os.environ[b2_alias]
        return
    try:
        from google.colab import userdata  # type: ignore[import-not-found]
        value = userdata.get(env_var) or (b2_alias and userdata.get(b2_alias))
        if value:
            os.environ[env_var] = value
            return
    except Exception:
        pass
    if prompt:
        os.environ[env_var] = getpass.getpass(prompt)


PRIVATE_B2_BUCKET = os.environ.get("PRIVATE_B2_BUCKET", "").strip()
PRIVATE_B2_REGION = os.environ.get("PRIVATE_B2_REGION", "").strip()
PRIVATE_B2_PREFIX = os.environ.get("PRIVATE_B2_PREFIX", "whisper-runs").strip("/")

if not PRIVATE_B2_BUCKET:
    PRIVATE_B2_BUCKET = input(
        "Optional: name of your private B2 bucket for writing transcripts "
        "(leave blank to skip the upload step): "
    ).strip()

if PRIVATE_B2_BUCKET:
    if not PRIVATE_B2_REGION:
        PRIVATE_B2_REGION = input(
            "Region of your private B2 bucket (e.g. us-west-001, us-east-005): "
        ).strip()
    _resolve_credential(
        "AWS_ACCESS_KEY_ID",
        b2_alias="B2_APPLICATION_KEY_ID",
        prompt="AWS_ACCESS_KEY_ID (B2 application key id): ",
    )
    _resolve_credential(
        "AWS_SECRET_ACCESS_KEY",
        b2_alias="B2_APPLICATION_KEY",
        prompt="AWS_SECRET_ACCESS_KEY (B2 application key): ",
    )
    PRIVATE_ENDPOINT = f"https://s3.{PRIVATE_B2_REGION}.backblazeb2.com"
    print(
        f"Will write transcript to s3://{PRIVATE_B2_BUCKET}/{PRIVATE_B2_PREFIX}/ on {PRIVATE_ENDPOINT}"
    )
else:
    PRIVATE_ENDPOINT = None
    print("No private bucket given; transcript will be printed only.")

## Stream the Audio from B2

Whisper's `transcribe()` accepts a path on disk. We pull the audio out of the public B2 bucket with an anonymous `boto3` S3 client and write it to a temp file. The client pins `region_name` to `us-west-001` (`b2datasets`'s home region) and sets a custom `user_agent_extra` so the request is attributable on B2's server-side logs even though the call itself is unsigned.

In [ ]:
import tempfile
from pathlib import Path

import boto3
from botocore import UNSIGNED
from botocore.config import Config

public_s3 = boto3.client(
    "s3",
    endpoint_url=PUBLIC_ENDPOINT,
    region_name=PUBLIC_REGION,
    config=Config(
        signature_version=UNSIGNED,
        user_agent_extra="b2-notebook-whisper",
    ),
)

audio_path = Path(tempfile.gettempdir()) / "jfk.flac"
public_s3.download_file(PUBLIC_BUCKET, AUDIO_KEY, str(audio_path))

print(f"Downloaded {audio_path.stat().st_size:,} bytes -> {audio_path}")

### Alternative: Plain HTTPS Download

Because the dataset bucket is configured as **Public** in B2, its files are also reachable via plain HTTPS without any S3 SDK. Useful for quick `curl` / `wget` checks or for any HTTP-aware tool:

In [ ]:
PUBLIC_HTTPS_URL = (
    f"https://{PUBLIC_BUCKET}.s3.us-west-001.backblazeb2.com/{AUDIO_KEY}"
)
print(f"Friendly subdomain URL: {PUBLIC_HTTPS_URL}")

## Transcribe with Whisper

Load Whisper and transcribe the local file. Default model is `tiny` (smallest, fastest, ~75 MB) which is plenty for the JFK clip; switch via the `WHISPER_MODEL_SIZE` env var to `base`, `small`, `medium`, or `large` for higher accuracy at the cost of longer runtime and more memory.

In [ ]:
import whisper

model_size = os.environ.get("WHISPER_MODEL_SIZE", "tiny")
print(f"Loading Whisper model: {model_size}")
model = whisper.load_model(model_size)
result = model.transcribe(str(audio_path))

print("\nDetected language:", result.get("language"))
print("\nTranscript:\n")
print(result["text"].strip())

## (Optional) Persist the Transcript Back to B2

If you supplied a private bucket and credentials above, upload the full Whisper result (text + segments with timestamps + detected language) as a JSON object. Skipped otherwise. The write client pins `region_name=PRIVATE_B2_REGION` and sets the same `user_agent_extra` so writes are attributable on B2.

In [ ]:
import json
from datetime import datetime, timezone

if PRIVATE_B2_BUCKET and PRIVATE_ENDPOINT:
    private_s3 = boto3.client(
        "s3",
        endpoint_url=PRIVATE_ENDPOINT,
        region_name=PRIVATE_B2_REGION,
        config=Config(
            signature_version="s3v4",
            user_agent_extra="b2-notebook-whisper",
        ),
    )
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    out_key = f"{PRIVATE_B2_PREFIX}/{timestamp}-jfk.json"

    private_s3.put_object(
        Bucket=PRIVATE_B2_BUCKET,
        Key=out_key,
        Body=json.dumps(result, indent=2).encode("utf-8"),
        ContentType="application/json",
    )
    print(f"Wrote transcript -> s3://{PRIVATE_B2_BUCKET}/{out_key}")
else:
    print("Skipped upload: no PRIVATE_B2_BUCKET configured.")

## Summary

### What We Accomplished

&check; **Streamed an audio file from a public B2 bucket** using an anonymous boto3 client (no application key needed for the input).  
&check; **Transcribed it with OpenAI Whisper** running locally on CPU in seconds.  
&check; **Persisted the transcript JSON back to a private B2 bucket** using a second, authenticated boto3 client.

### Bucket Layout

```
s3://b2datasets/whisper-demo/
    jfk.flac                       (public, anonymously readable)

s3://<your-bucket>/whisper-runs/
    20260522T120000Z-jfk.json      (transcript JSON written by this notebook)
```

### Why B2

- **One S3-compatible API** for both the public dataset and your private outputs; only the credentials change between the two clients.
- **Free egress** to Bandwidth Alliance partners and free egress equal to monthly storage everywhere else, so reading large audio archives for batch transcription stays inexpensive.
- **Public buckets** are also reachable over plain HTTPS, so the same files work with `curl`, browsers, and any HTTP-aware library with no SDK required.

### Next Steps

- Swap `WHISPER_MODEL_SIZE=base` (or `small` / `medium` / `large`) for higher transcription accuracy.
- Point `AUDIO_KEY` at your own audio archive and loop over multiple keys to batch-transcribe.
- Wire the upload step into a B2 event-notification webhook to auto-transcribe new uploads.